# Circular Pad — Curved Geometry and Palace Eigenmode

Canonical source-only candidate; semantic state CONVERGING, delivery
LOCAL CANDIDATE, no test writes. OrPen owns physical source facts, this
notebook owns case settings and final Nets, and SCGSim owns runtime.

This synthetic `circular_pad_capacitor` linear LC model uses radius 100
µm, radial gap 20 µm, a 1000 µm die, current PDK 100 nm Al and 500 µm
Si. The east-facing nonmetal locator binds the linear 10 nH inductive
branch, with no added capacitive branch. It is not ordinary CPW, generic
RLC or a measured device. Component source owns curved boundary chains
and local PAD/GROUND facts; this notebook owns final Nets.

Read the [eigenmode
requirements](../../../simulation/requirements/eigenmode.qmd) and
[EPR/loss requirements](../../../simulation/requirements/epr-loss.qmd).
The authoring target is [SCGSim v2
RC1](https://github.com/OrPenStrike/scgsim/tree/85d8bedb733b1e17737d935029618351b30ea5de)
in a separate consumer environment; ordinary project dependencies and
pins remain unchanged.

The default `AUTHORING_ONLY` action constructs the source coupon and
detached Plan/Sim inputs only. It never meshes, generates config,
prepares a native handoff or solves. Heavy preparation requires explicit
`prepare_handoff` or `run`. Quarto execution remains disabled.
LiteratureCentral MA/MS/SA films are synthetic assumptions; numerical
controls are inputs, not adequacy Gates. No new native geometry, mesh,
fields, solve or numerical-accuracy claim is made.

Historical evidence is separate: the original
`circular_pad_palace_eigenmode_20261007_08` cohort used SCGSim 1.1.0rc1
and reported 5.573354161115 GHz with retained PCG warnings. Those
results do not execute or validate this source; see the [original
evidence and identity
manifest](https://github.com/OrPenStrike/orpen-sc-pdk/blob/develop/notebooks/ComponentSimulation/CircularPad/evidence/README.md).

In [ ]:
import importlib.metadata
import json
import os
import subprocess
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.palace import reanalyze_epr, resolve_epr_result, show_epr

WORKFLOW_ACTION = "AUTHORING_ONLY"
# AUTHORING_ONLY | prepare_handoff | run | analyze_handoff | save_from_returned | analyze_saved_epr
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    import gdsfactory as gf
    from scgsim.geometry import GeometryPlan
    from scgsim.palace import EigenmodeSim

    import orpen_sc_pdk
    from orpen_sc_pdk import (
        get_interface_preset_records,
        get_material_records,
        validate_interface_preset_records,
    )
    from orpen_sc_pdk.tech import get_single_die_layer_stack

if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    from scgsim.palace import inspect_run_trustworthiness, resolve_palace_result

RUN_ID = f"circular_pad_palace_eigenmode_v2_{uuid4().hex}"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "CircularPad"
RUN_ROOT = OUTPUT_ROOT / RUN_ID
if WORKFLOW_ACTION not in {
    "AUTHORING_ONLY",
    "prepare_handoff",
    "run",
    "analyze_handoff",
    "save_from_returned",
    "analyze_saved_epr",
}:
    raise ValueError("Unknown WORKFLOW_ACTION")
display({"scgsim_version": importlib.metadata.version("scgsim"), "run_id": RUN_ID})

## Build Component Coupon

In [ ]:
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    OUTER_RADIUS_UM = 100.0
    INNER_RADIUS_UM = 0.0
    GAP_UM = 20.0
    DIE_SIDE_UM = 1000.0
    JUNCTION_WIDTH_UM = 2.0
    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(
        "circular_pad_capacitor",
        outer_radius_um=OUTER_RADIUS_UM,
        inner_radius_um=INNER_RADIUS_UM,
        gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM,
        junction_width_um=JUNCTION_WIDTH_UM,
    )
    layer_stack = get_single_die_layer_stack()
    plan = GeometryPlan(
        component=component,
        root_id="chip",
        layer_stack=layer_stack,
        material_records=get_material_records(),
        coupon_padding_um=0.0,
    )
    plan.set_nets({"pad": ("chip/PAD",), "ground": ("chip/GROUND",)})
    plan.set_vacuum_region(padding=(250.0, 250.0, 1000.0))
    snapshot = plan.prepare()
    display(component)
    display(
        {
            "bbox_um": str(component.dbbox()),
            "ports": component.ports,
            "paired_gds_sha256": snapshot.gds_sha256,
        }
    )

## Configure EPR / Problem

LiteratureCentral MA/MS/SA are synthetic model conventions, not measured
process data. EPR is baseline unmasked / zero-inset only. Positive-inset
perimeter distances use chords in the selected Palace runtime; this
workflow does not claim exact curved mask separation or introduce a mask
study.

In [ ]:
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    ROUTE = "B"
    sim = EigenmodeSim()
    sim.set_plan(snapshot)
    sim.set_output_dir(RUN_ROOT)
    records = validate_interface_preset_records(get_interface_preset_records())
    preset_names = {kind: f"LiteratureCentral_{kind}" for kind in ("MA", "MS", "SA")}
    interface_presets = {kind: records[name] for kind, name in preset_names.items()}
    sim.set_surface_epr(
        representation=ROUTE,
        specs={
            kind: {
                "thickness": record["thickness"],
                "permittivity": record["permittivity"],
                "loss_tangent": record["loss_tangent"],
                "source": record["source"],
                "preset": preset_names[kind],
            }
            for kind, record in interface_presets.items()
        },
    )
    PORT_NAME = "chip/o_junction_lumped"
    PORT_INDUCTANCE_H = 10e-9
    PORT_CAPACITANCE_F = 0.0  # Inductive-only model; no Palace capacitive branch.
    sim.add_port(PORT_NAME, layer="D0_TOP_M1", layout_sheet=True, inductance=PORT_INDUCTANCE_H)
    sim.set_epr_request(
        surface_interfaces=("MA", "MS", "SA"), bulk_domain_ids=None, port_names=(PORT_NAME,)
    )
    NUM_MODES = 1
    SEARCH_FREQUENCY_HZ = 1e9  # Search lower bound, not a desired frequency.
    EIGENMODE_TOLERANCE = 1e-6
    SAVE_FIELDS = 1
    sim.set_eigenmode(
        num_modes=NUM_MODES,
        target=SEARCH_FREQUENCY_HZ,
        tolerance=EIGENMODE_TOLERANCE,
        save=SAVE_FIELDS,
    )
    display(
        {
            "route": ROUTE,
            "inductance_h": PORT_INDUCTANCE_H,
            "capacitance_f": PORT_CAPACITANCE_F,
            "interface_presets": interface_presets,
        }
    )
    display(sim.input_summary())

## Build Mesh

Geometric order is independent of field FEM order. SCGSim owns the
shared curve arrangement, source rebinding, elevation and one HighOrder
optimization.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    REFINED_MESH_SIZE_UM = 10.0
    MAX_MESH_SIZE_UM = 200.0
    ALGORITHM_3D = "HXT"
    MESH_THREADS = 8
    SURFACE_THREADS = 1
    GEOMETRY_ORDER = 2
    HIGH_ORDER_OPTIMIZE = True
    sim.set_mesh(
        refined_mesh_size=REFINED_MESH_SIZE_UM,
        max_mesh_size=MAX_MESH_SIZE_UM,
        algorithm_3d=ALGORITHM_3D,
        threads=MESH_THREADS,
        surface_threads=SURFACE_THREADS,
        geometry_order=GEOMETRY_ORDER,
        high_order_optimize=HIGH_ORDER_OPTIMIZE,
    )
    MESH_PATH = sim.mesh()
    display(MESH_PATH)
    display(sim.mesh_summary())

## Generate Config

The maximum permits native early stopping; four completed AMR passes are
not required. Controls are declared inputs, not scientific acceptance
criteria.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    FEM_ORDER = 2
    LINEAR_TOLERANCE = 1e-6
    MAX_ITERATIONS = 2000
    AMR_MAX_PASSES = 4
    AMR_TOLERANCE = 0.02
    AMR_UPDATE_FRACTION = 0.15
    AMR_NONCONFORMAL = False
    SAVE_ADAPT_ITERATIONS = True
    sim.set_numerical(
        order=FEM_ORDER,
        tolerance=LINEAR_TOLERANCE,
        max_iterations=MAX_ITERATIONS,
        solver_type="Default",
        preconditioner="Default",
        device="CPU",
        amr_max_passes=AMR_MAX_PASSES,
        amr_tolerance=AMR_TOLERANCE,
        amr_update_fraction=AMR_UPDATE_FRACTION,
        amr_nonconformal=AMR_NONCONFORMAL,
        save_adapt_iterations=SAVE_ADAPT_ITERATIONS,
        output_paraview=True,
        output_grid_function=False,
    )
    CONFIG_PATH = sim.write_config()
    display(CONFIG_PATH)
    display(sim.mesh_summary())

## Prepare Handoff

Only `run` invokes the generated script. Use a fresh RUN_ID for each new
preparation; existing destinations fail rather than being overwritten.
Exact host executable/setup provenance belongs in the owner’s execution
receipt, not a machine-local public default. There is no automatic retry
or fallback. `PALACE_EXECUTABLE` must select an actual Palace MPI
wrapper: the generated script supplies `-np 8 -nt 1` through
`resources["command_style"] = "wrapper"`. A direct ELF is not a wrapper.
Select a fresh RUN_ID before changing the action to `prepare_handoff` or
`run`; the fresh RUN_ID never names historical evidence.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    MACHINE_PROFILE = "direct-local"
    PALACE_EXECUTABLE = os.environ["PALACE_EXECUTABLE"]
    SETUP_COMMANDS = tuple(os.environ.get("PALACE_SETUP_COMMANDS", "").splitlines())
    RESOURCES = {"processes": 8, "threads": 1, "command_style": "wrapper"}
    HANDOFF = sim.prepare_handoff(
        profile=MACHINE_PROFILE,
        executable=PALACE_EXECUTABLE,
        resources=RESOURCES,
        setup_commands=SETUP_COMMANDS,
    )
    display(HANDOFF)
    display({"retain_independently": HANDOFF.handoff_id})
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Analyze Returned Run

In a fresh analysis kernel, set `CIRCULAR_PAD_RETURNED_RUN_DIR` to the
actual returned folder and supply its independently retained preparation
ID through `CIRCULAR_PAD_PALACE_HANDOFF_ID`. There is no historical-path
or identity default. Missing inputs fail natively; failed/partial
results stay failed/partial and never become a strict completed result.
Analyze does not build, mesh or solve.

### Run Identity

### Numerical Evidence

### Simulation Benchmark

In [ ]:
if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    RETURNED_RUN_DIR = Path(os.environ["CIRCULAR_PAD_RETURNED_RUN_DIR"])
    EXPECTED_HANDOFF_ID = os.environ["CIRCULAR_PAD_PALACE_HANDOFF_ID"]
    report = inspect_run_trustworthiness(RETURNED_RUN_DIR)
    display(report.show_run_trustworthiness(show_details=True))
    display(report.show_simulation_benchmark(show_details=True))
    if WORKFLOW_ACTION == "save_from_returned" or report.completeness == "complete":
        result = resolve_palace_result(RETURNED_RUN_DIR, expected_handoff_id=EXPECTED_HANDOFF_ID)

### Physics Analysis Results

Historical `_08` only: the frequency moves from 5.310657080530 to
5.573354161115 GHz across the recorded refinements; the configured
four-refinement maximum was reached. The final native indicator is
0.1186681863426 versus the configured 0.02; this is an observation, not
a numerical-adequacy criterion. All five postprocessing phases emitted
two PCG nonconvergences at 10000 iterations (final relative residuals
9.537e-6 and 2.349e-5). The initial natural PMC/ZeroCharge boundary
warning also remains part of the run evidence. The typed EPR result
contains nine surface, two bulk and one junction row. The raw signed
Palace junction participation is -0.9948401783541
(`signed_palace=true`): it is neither absolute-valued nor interpreted as
a negative physical energy. Surface values are unmasked, zero-inset
baselines; the interface losses are synthetic model inputs. See the
[derived evidence tables and
provenance](https://github.com/OrPenStrike/orpen-sc-pdk/blob/develop/notebooks/ComponentSimulation/CircularPad/evidence/README.md).

In [ ]:
if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    if report.completeness == "complete":
        display(result.show_physics_quantities(ranking_limit=None))
        epr = result.epr_result()
        display(show_epr(epr, mode=1))
    else:
        display(report.show_physics_quantities(ranking_limit=None))

### Saved EPR and Offline Loss

Strict completed result extraction and loss reanalysis require no new
component, GDS, CAD, mesh or fields. Empty overrides preserve the
verified assumptions; explicit caller film edits change offline loss
assumptions, not native fields. Use real Group IDs in the portable
snapshot. Do not absolute-value signed EPR. The caller-owned save
destination is opened exclusively to avoid overwrite.

In [ ]:
if WORKFLOW_ACTION == "save_from_returned":
    EPR_SAVE_PATH = Path(os.environ["CIRCULAR_PAD_PALACE_EPR_SAVE_PATH"])
    epr = result.epr_result()
    with EPR_SAVE_PATH.open("x", encoding="utf-8") as stream:
        json.dump(epr.to_payload(), stream, indent=2)

Set `WORKFLOW_ACTION = "analyze_saved_epr"` in a fresh kernel to reuse a
specified snapshot. This action reads only the saved EPR path; it
requires no returned run directory, handoff ID, component, GeometryPlan,
mesh or fields. `save_from_returned` creates a new path exclusively from
an actual strict returned result. Existing files are never silently
reused as a save fallback. `analyze_handoff` does not create a snapshot.

In [ ]:
if WORKFLOW_ACTION in {"save_from_returned", "analyze_saved_epr"}:
    EPR_SAVE_PATH = Path(os.environ["CIRCULAR_PAD_PALACE_EPR_SAVE_PATH"])
    saved_epr = resolve_epr_result(EPR_SAVE_PATH)
    SURFACE_DEFAULTS = {}  # Explicit MA/MS/SA film assumptions, if requested.
    GROUP_OVERRIDES = {}  # Real saved Group IDs only.
    offline_epr = reanalyze_epr(
        saved_epr,
        surface_defaults=SURFACE_DEFAULTS,
        group_overrides=GROUP_OVERRIDES,
    )
    display(show_epr(offline_epr, mode=1))

### Simulation Performance / Benchmarks

Read actual emitted geometric node/order data and native AMR histories
beside the declared controls. The final native metadata records 846008
ND unknowns and 114197 tetrahedra, using 8 MPI processes × 1 OpenMP
thread. Ordinary saved VTU output across eight pieces contains 10-node
Lagrange tetrahedra (VTK 71) and 6-node Lagrange triangles (VTK 69).
That observes quadratic serialization after AMR, not exact shared-node
motion or geometric accuracy. Continuous-circle fidelity and interior
nodal state beyond those saved outputs remain unmeasured. No Q/T1,
cross-solver equivalence, accuracy or convergence conclusion follows.